DEPENDICIES

In [11]:
pip install -r requirements.txt

1055.23s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached pandas-3.0.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.2-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp314-cp314-macosx_10_15_universal2.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.3 kB)
  Using cached pillow-12.1.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.met

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

GATHER DATASETS

In [13]:
test_df = pd.read_csv('test.csv')

EDA

In [14]:
print("\nTest set structure:")
print(test_df.head(10))

# Count pairs per query image
query_counts = test_df['query_image'].value_counts()
print(f"\nPairs per query image: {query_counts.iloc[0]}")

# Verify total pairs
n_images = len(set(test_df['query_image']))
expected_pairs = n_images * (n_images - 1)
print(f"Total images: {n_images}")
print(f"Expected pairs: {expected_pairs:,}")
print(f"Actual pairs: {len(test_df):,}")


Test set structure:
   row_id    query_image  gallery_image
0       0  test_0001.png  test_0002.png
1       1  test_0001.png  test_0003.png
2       2  test_0001.png  test_0004.png
3       3  test_0001.png  test_0005.png
4       4  test_0001.png  test_0006.png
5       5  test_0001.png  test_0007.png
6       6  test_0001.png  test_0008.png
7       7  test_0001.png  test_0009.png
8       8  test_0001.png  test_0010.png
9       9  test_0001.png  test_0011.png

Pairs per query image: 370
Total images: 371
Expected pairs: 137,270
Actual pairs: 137,270


TRAINING THE MODEL

PREDICTION

In [15]:
# Load model
with torch.no_grad():
    model = JaguarReIDModel.load('jaguar_reid_model.pth')

    # Get unique test images
    unique_images = sorted(set(test_df['query_image']) | set(test_df['gallery_image']))

    # Extract embeddings for all test images
    print("Extracting embeddings...")
    embeddings = {}
    for img_file in unique_images:
        img_path = TEST_DIR / img_file
        embedding = model.extract_embedding(img_path)
        embeddings[img_file] = embedding

    # Compute similarities for all test pairs
    print("Computing similarities...")
    similarities = []
    for _, row in test_df.iterrows():
        query_emb = embeddings[row['query_image']]
        gallery_emb = embeddings[row['gallery_image']]

        # Cosine similarity
        sim = np.dot(query_emb, gallery_emb) / (
            np.linalg.norm(query_emb) * np.linalg.norm(gallery_emb)
        )

        # Map from [-1, 1] to [0, 1]
        sim = (sim + 1) / 2

        similarities.append(sim)

    print(f"Generated {len(similarities)} predictions")


NameError: name 'JaguarReIDModel' is not defined

CREATE SUBMISSION

In [ ]:
submission = pd.DataFrame({
    'row_id': test_df['row_id'],
    'similarity': similarities
})

# Validate
assert len(submission) == 137270
assert submission['row_id'].tolist() == list(range(137270))
assert (submission['similarity'] >= 0).all()
assert (submission['similarity'] <= 1).all()

print("\nSubmission statistics:")
print(submission['similarity'].describe())

VALIDATION AND SUBMISSION

In [ ]:
validate_submission('my_submission.csv')

# Save submission
submission.to_csv('my_submission.csv', index=False)
print("\nSubmission saved: my_submission.csv")
print(f"File size: {Path('my_submission.csv').stat().st_size / 1024 / 1024:.2f} MB")